In [ ]:
import h5py
import utils as wbi
from utils import preprocess, calFlow3d_Wei_v1, visualization,mask,option,IO,reference
import numpy as np
import nd2
import tifffile as tiff
from matplotlib import pyplot as plt

#set the option

thresFactor=5
maskRange=[5,2000]
smoothPenalty_raw=0.05
option['r']=5
option['layer']=1
option['iter']=10

# The params which can control mask
# maskRange: For the raw image, we only mask the region in the maskRange
# threFactor: the times of std 
frames=range(0,500)

movingFilePath="/home/cyf/wbi/Virginia/f2013/250705_f2013_ubi_gcamp7f_bactin_mcherry_6dpf_15849.nd2"
refenceFilePath_start='/home/cyf/wbi/Virginia/f2013/250705_f2013_ubi_gcamp7f_bactin_mcherry_6dpf_15846.nd2'
refenceFilePath_end='/home/cyf/wbi/Virginia/f2013/250705_f2013_ubi_gcamp7f_bactin_mcherry_6dpf_15850.nd2'

meta_start=IO.readMeta(refenceFilePath_start)
meta_end=IO.readMeta(refenceFilePath_end)
meta_moving=IO.readMeta(movingFilePath)
dat_ref_begin=IO.readFrame(refenceFilePath_start,frame=0,channel=1)
dat_ref_end=IO.readFrame(refenceFilePath_end,frame=0,channel=1)


Z ratio is 3.0769230769230766
Data size is (1152, 1152, 19)
Total frames is 2
Z ratio is 3.0769230769230766
Data size is (1152, 1152, 19)
Total frames is 3
Z ratio is 3.0769230769230766
Data size is (1152, 1152, 1)
Total frames is 47997


In [ ]:
import tifffile as tiff
import numpy as np
import imageio

MostPairedSlice=11

ref_stack=dat_ref_begin[:,:,MostPairedSlice]
dat_ref=np.stack([ref_stack]*3,axis=2)

ref_stack2=dat_ref_end[:,:,MostPairedSlice]
dat_ref_2=np.stack([ref_stack2]*3,axis=2)

dat_channel1 = []
dat_channel2 = []
errors = []
concat_images = []
concat_images_with_checks = []

option['mask_ref']=mask.getMask(dat_ref,thresFactor)
option['mask_ref']=mask.bwareafilt3_wei(option['mask_ref'],maskRange)

dat_ref=preprocess.normalize_to_255(dat_ref)
Pnltfactor = preprocess.getSmPnltNormFctr(dat_ref, option)
smoothPenalty=Pnltfactor*smoothPenalty_raw

option['motion']=np.zeros([ref_stack.shape[0],ref_stack.shape[1],2,3])

for i in range(500):
    ca_stack = IO.readFrame(movingFilePath,frame=i,channel=0)
    mov_stack = IO.readFrame(movingFilePath,frame=i,channel=1)
    # fake 3D stack
    dat_mov = np.stack([mov_stack] * 3, axis=2)
    dat_ca = np.stack([ca_stack] * 3, axis=2)
    
    option['mask_mov'] = mask.getMask(dat_mov, thresFactor)
    option['mask_mov'] = mask.bwareafilt3_wei(option['mask_mov'], maskRange)
    dat_mov=preprocess.normalize_to_255(dat_mov)
    motion_current, error, new_coords = calFlow3d_Wei_v1.getMotion(dat_mov, dat_ref, smoothPenalty, option)
    errors.append(error)
    corrected_ca = calFlow3d_Wei_v1.correctMotion(dat_ca, motion_current)
    corrected_mov = calFlow3d_Wei_v1.correctMotion(dat_mov, motion_current)

    # Create difference check image
    diff_check = np.abs(corrected_mov[:, :, 0] - dat_ref[:, :, 0])

    print(f"Frame: {i+1}\tInitial Error is:{np.mean((dat_mov-dat_ref)**2)}\tEventual Error: {np.mean((corrected_mov-dat_ref)**2)}")
  
    dat_channel1.append(corrected_ca[:, :, 0])
    dat_channel2.append(corrected_mov[:, :, 0])
    concat = np.concatenate([dat_mov[:,:,0], dat_channel2[i]], axis=1)
    # visulization.visualize_2d_image(concat,figsize=(12,4))
    # visulization.visualize_2d_image((corrected_mov-dat_ref)[:,:,0],figsize=(6,6),title='dat_mov after correction')
    concat_images.append(concat.astype(np.float32))
    concat_with_checks = np.concatenate([
        dat_mov[:, :, 0].astype(np.float32),
        corrected_mov[:, :, 0].astype(np.float32),
        dat_ref[:, :, 0].astype(np.float32),
        diff_check.astype(np.float32)
    ], axis=1)
    concat_images_with_checks.append(concat_with_checks)
    
concat_images_uint8 = [img.astype(np.uint8) for img in concat_images]


Frame: 1	Initial Error is:1630.997846272109	Eventual Error: 996.1112321456218
Frame: 2	Initial Error is:1623.1107817917139	Eventual Error: 999.9735640462112
Frame: 3	Initial Error is:1636.8791325812238	Eventual Error: 997.2581745874971
Frame: 4	Initial Error is:1634.608448298424	Eventual Error: 1002.7342567539868
Frame: 5	Initial Error is:1594.8397666214294	Eventual Error: 980.6833427214742
Frame: 6	Initial Error is:1606.6926580691238	Eventual Error: 985.2822700922688
Frame: 7	Initial Error is:1600.270072196206	Eventual Error: 986.3163046939806
Frame: 8	Initial Error is:1597.624098667162	Eventual Error: 985.2359063584662
Frame: 9	Initial Error is:1604.3849808901737	Eventual Error: 985.1522030650266
Frame: 10	Initial Error is:1646.097080784661	Eventual Error: 996.9924448404057
Frame: 11	Initial Error is:1593.418764957173	Eventual Error: 984.9043725712504
Frame: 12	Initial Error is:1619.9845132763676	Eventual Error: 986.0362584435298
Frame: 13	Initial Error is:1591.7374155850932	Eventual

In [ ]:
dat_channel1_array=np.array(dat_channel1)
dat_channel2_array=np.array(dat_channel2)
concat_images_array=np.array(concat_images)

tiff.imwrite("dat_channel1_corrected_0805.tiff", dat_channel1_array.transpose(2,0,1).astype(np.float32))
tiff.imwrite("dat_channel2_corrected_0805.tiff", dat_channel2_array.transpose(2,0,1).astype(np.float32))
tiff.imwrite("comparison_concat_0805.tiff",concat_images_array[:,:,:])  # 每帧原图+配准图

with open("registration_errors_0805.txt", "w") as f:
    for i, err in enumerate(errors):
        f.write(f"Frame {i+1}: Error = {err:.6f}\n")